# Notebook 11 — Exercises: Practice Problems

This notebook contains **2–4 practice exercises per topic** covering all material from Notebooks 01–10. Each exercise includes:

1. **Problem statement** with embedded SMILES data (no external files needed)
2. **Hints** to nudge you in the right direction
3. **Empty solution cell** for you to fill in
4. **Reference solution** you can reveal when ready

All data is self-contained — just run the imports cell and start solving.

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import (
    AllChem, Descriptors, rdMolDescriptors, Draw,
    MACCSkeys, rdFingerprintGenerator, FilterCatalog,
    rdFMCS, rdChemReactions, PandasTools,
)
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.Draw import rdMolDraw2D, IPythonConsole
from rdkit.ML.Cluster import Butina
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit.Chem.inchi import MolToInchi, InchiToInchiKey
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

---

## Topic 1: Molecular Representations (Notebook 01)

### Exercise 1.1 — Write SMILES from Drug Names

**Problem:** Given these drug names, write valid SMILES strings and verify each by computing its molecular formula.

1. Ethanol
2. Acetic acid
3. Benzene
4. Paracetamol (acetaminophen)
5. Nicotine

**Hints:**
- Use `Chem.MolFromSmiles()` to parse and verify each SMILES — it returns `None` for invalid input.
- Use `rdMolDescriptors.CalcMolFormula()` to compute the molecular formula.
- Remember: aromatic rings in SMILES use lowercase letters (`c1ccccc1` for benzene).
- Nicotine has a pyridine ring fused to an N-methylpyrrolidine.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 1.1
drug_smiles = {
    "Ethanol": "CCO",
    "Acetic acid": "CC(=O)O",
    "Benzene": "c1ccccc1",
    "Paracetamol": "CC(=O)NC1=CC=C(O)C=C1",
    "Nicotine": "CN1CCC[C@H]1C1=CN=CC=C1",
}

for name, smi in drug_smiles.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        formula = rdMolDescriptors.CalcMolFormula(mol)
        canonical = Chem.MolToSmiles(mol)
        print(f"{name:15s} | SMILES: {canonical:35s} | Formula: {formula}")
    else:
        print(f"{name:15s} | INVALID SMILES: {smi}")

### Exercise 1.2 — Convert Between Representations

**Problem:** Given this SMILES for ibuprofen: `"CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"`

1. Convert to canonical SMILES
2. Generate InChI
3. Generate InChIKey
4. Convert the InChI back to a molecule and verify the SMILES matches the canonical form

**Hints:**
- `Chem.MolToSmiles(mol)` returns canonical SMILES.
- `MolToInchi(mol)` generates the InChI string.
- `InchiToInchiKey(inchi)` generates the InChIKey.
- `Chem.MolFromInchi(inchi)` converts InChI back to a molecule object.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 1.2
ibuprofen_smi = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
mol = Chem.MolFromSmiles(ibuprofen_smi)

# 1. Canonical SMILES
canonical = Chem.MolToSmiles(mol)
print(f"Canonical SMILES: {canonical}")

# 2. InChI
inchi = MolToInchi(mol)
print(f"InChI:            {inchi}")

# 3. InChIKey
inchi_key = InchiToInchiKey(inchi)
print(f"InChIKey:         {inchi_key}")

# 4. Round-trip: InChI -> mol -> SMILES
from rdkit.Chem.inchi import MolFromInchi
mol_from_inchi = MolFromInchi(inchi)
roundtrip_smi = Chem.MolToSmiles(mol_from_inchi)
print(f"\nRound-trip SMILES: {roundtrip_smi}")
print(f"Matches original:  {canonical == roundtrip_smi}")

### Exercise 1.3 — Identify Invalid SMILES

**Problem:** Which of these SMILES are valid? For valid ones, print their molecular formula.

```python
smiles_list = ["CCO", "C(C)(C)(C)(C)C", "c1ccccc1", "XYZ", "C#C#C#C", "[Cu+2]", "C1=CC=CC=C1N(=O)=O"]
```

**Hints:**
- `Chem.MolFromSmiles()` returns `None` for invalid SMILES.
- Some SMILES may be syntactically valid but represent unusual chemistry (e.g., pentavalent carbon).
- Use `rdMolDescriptors.CalcMolFormula()` for the formula.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 1.3
smiles_list = ["CCO", "C(C)(C)(C)(C)C", "c1ccccc1", "XYZ", "C#C#C#C", "[Cu+2]", "C1=CC=CC=C1N(=O)=O"]

for smi in smiles_list:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        formula = rdMolDescriptors.CalcMolFormula(mol)
        print(f"  VALID   | {smi:30s} | Formula: {formula}")
    else:
        print(f"  INVALID | {smi:30s} |")

---

## Topic 2: Molecular Visualization (Notebook 02)

### Exercise 2.1 — Draw Amino Acid Grid with Names

**Problem:** Create a grid image of these amino acids with their names as legends. Display as a 3x2 grid.

```python
amino_acids = {
    "Glycine": "NCC(=O)O",
    "Alanine": "CC(N)C(=O)O",
    "Valine": "CC(C)C(N)C(=O)O",
    "Leucine": "CC(C)CC(N)C(=O)O",
    "Phenylalanine": "NC(CC1=CC=CC=C1)C(=O)O",
    "Tryptophan": "NC(CC1=CNC2=CC=CC=C12)C(=O)O",
}
```

**Hints:**
- `Draw.MolsToGridImage()` accepts `molsPerRow`, `legends`, and `subImgSize` parameters.
- Build a list of mol objects and a list of name strings in the same order.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 2.1
amino_acids = {
    "Glycine": "NCC(=O)O",
    "Alanine": "CC(N)C(=O)O",
    "Valine": "CC(C)C(N)C(=O)O",
    "Leucine": "CC(C)CC(N)C(=O)O",
    "Phenylalanine": "NC(CC1=CC=CC=C1)C(=O)O",
    "Tryptophan": "NC(CC1=CNC2=CC=CC=C12)C(=O)O",
}

mols = [Chem.MolFromSmiles(smi) for smi in amino_acids.values()]
names = list(amino_acids.keys())

Draw.MolsToGridImage(mols, molsPerRow=3, legends=names, subImgSize=(300, 250))

### Exercise 2.2 — Highlight Amine Groups in Amino Acids

**Problem:** Use the SMARTS pattern `[NX3H2]` to find and highlight the primary amine group in each amino acid from Exercise 2.1. Draw a grid with the matched atoms highlighted.

**Hints:**
- `Chem.MolFromSmarts()` creates a query pattern.
- `mol.GetSubstructMatch(pattern)` returns matching atom indices.
- `Draw.MolsToGridImage()` accepts `highlightAtomLists` — a list of lists of atom indices, one per molecule.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 2.2
amino_acids = {
    "Glycine": "NCC(=O)O",
    "Alanine": "CC(N)C(=O)O",
    "Valine": "CC(C)C(N)C(=O)O",
    "Leucine": "CC(C)CC(N)C(=O)O",
    "Phenylalanine": "NC(CC1=CC=CC=C1)C(=O)O",
    "Tryptophan": "NC(CC1=CNC2=CC=CC=C12)C(=O)O",
}

amine_pattern = Chem.MolFromSmarts("[NX3H2]")
mols = [Chem.MolFromSmiles(smi) for smi in amino_acids.values()]
names = list(amino_acids.keys())

# Find matching atoms for highlighting
highlight_atoms = []
for mol in mols:
    matches = mol.GetSubstructMatches(amine_pattern)
    # Flatten all match tuples into a single list of atom indices
    atoms = [idx for match in matches for idx in match]
    highlight_atoms.append(atoms)

Draw.MolsToGridImage(
    mols, molsPerRow=3, legends=names,
    highlightAtomLists=highlight_atoms, subImgSize=(300, 250)
)

---

## Topic 3: Molecular Descriptors (Notebook 03)

### Exercise 3.1 — Compute Descriptor Profiles

**Problem:** For these molecules, compute MW, LogP, TPSA, HBD, HBA, and number of rotatable bonds. Create a pandas DataFrame and answer: which drug has the highest LogP? Which has the lowest TPSA?

```python
drugs = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Atorvastatin": "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
}
```

**Hints:**
- `Descriptors.MolWt()`, `Descriptors.MolLogP()`, `Descriptors.TPSA()`
- `rdMolDescriptors.CalcNumHBD()`, `rdMolDescriptors.CalcNumHBA()`
- `rdMolDescriptors.CalcNumRotatableBonds()`
- Use `pd.DataFrame` with a list of dicts, then `idxmax()` / `idxmin()` to find extremes.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 3.1
drugs = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Atorvastatin": "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
}

rows = []
for name, smi in drugs.items():
    mol = Chem.MolFromSmiles(smi)
    rows.append({
        "Name": name,
        "MW": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "TPSA": round(Descriptors.TPSA(mol), 2),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "RotBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
    })

df = pd.DataFrame(rows).set_index("Name")
display(df)

print(f"\nHighest LogP:  {df['LogP'].idxmax()} ({df['LogP'].max():.2f})")
print(f"Lowest TPSA:   {df['TPSA'].idxmin()} ({df['TPSA'].min():.2f})")

### Exercise 3.2 — Property Radar Chart

**Problem:** Create a radar (spider) chart comparing aspirin and atorvastatin across 5 normalized descriptors: MW, LogP, TPSA, HBD, HBA. Normalize each to the 0--1 range using these typical drug ranges:

| Descriptor | Min | Max |
|------------|-----|-----|
| MW         | 0   | 600 |
| LogP       | -2  | 7   |
| TPSA       | 0   | 200 |
| HBD        | 0   | 5   |
| HBA        | 0   | 10  |

```python
aspirin_smi = "CC(=O)OC1=CC=CC=C1C(=O)O"
atorvastatin_smi = "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1"
```

**Hints:**
- Use `matplotlib` polar projection: `fig, ax = plt.subplots(subplot_kw=dict(polar=True))`
- Compute angles: `angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)`
- Close the polygon by appending the first value to the end of each data list and the first angle to the angles array.
- `ax.fill()` for shading, `ax.plot()` for lines.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 3.2
aspirin_smi = "CC(=O)OC1=CC=CC=C1C(=O)O"
atorvastatin_smi = "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1"

# Normalization ranges: (min, max)
ranges = {"MW": (0, 600), "LogP": (-2, 7), "TPSA": (0, 200), "HBD": (0, 5), "HBA": (0, 10)}
categories = list(ranges.keys())

def compute_normalized(smi, ranges):
    mol = Chem.MolFromSmiles(smi)
    raw = {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
    }
    return [(raw[k] - ranges[k][0]) / (ranges[k][1] - ranges[k][0]) for k in categories]

asp_vals = compute_normalized(aspirin_smi, ranges)
ato_vals = compute_normalized(atorvastatin_smi, ranges)

# Build radar chart
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
# Close the polygon
asp_vals_closed = asp_vals + [asp_vals[0]]
ato_vals_closed = ato_vals + [ato_vals[0]]
angles_closed = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles_closed, asp_vals_closed, "o-", label="Aspirin", linewidth=2)
ax.fill(angles_closed, asp_vals_closed, alpha=0.15)
ax.plot(angles_closed, ato_vals_closed, "s-", label="Atorvastatin", linewidth=2)
ax.fill(angles_closed, ato_vals_closed, alpha=0.15)

ax.set_thetagrids(np.degrees(angles), categories)
ax.set_ylim(0, 1)
ax.set_title("Descriptor Radar: Aspirin vs. Atorvastatin", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

### Exercise 3.3 — Compute QED and Rank Drug-Likeness

**Problem:** Compute the QED (Quantitative Estimate of Drug-likeness) score for these molecules and rank them from most to least drug-like.

```python
molecules = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Diazepam": "CN1C(=O)CN=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C21",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Metformin": "CN(C)C(=N)NC(=N)N",
}
```

**Hints:**
- Import QED: `from rdkit.Chem.QED import qed`
- `qed(mol)` returns a float between 0 and 1.
- Use `sorted()` or `DataFrame.sort_values()` for ranking.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 3.3
from rdkit.Chem.QED import qed

molecules = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Diazepam": "CN1C(=O)CN=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C21",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Metformin": "CN(C)C(=N)NC(=N)N",
}

qed_scores = {}
for name, smi in molecules.items():
    mol = Chem.MolFromSmiles(smi)
    qed_scores[name] = round(qed(mol), 4)

# Sort by QED descending (most drug-like first)
ranked = sorted(qed_scores.items(), key=lambda x: x[1], reverse=True)

print("Rank | Drug         | QED Score")
print("-----|--------------|----------")
for i, (name, score) in enumerate(ranked, 1):
    print(f"  {i}  | {name:12s} | {score:.4f}")

---

## Topic 4: Substructure Searching (Notebook 04)

### Exercise 4.1 — Write SMARTS for Functional Groups

**Problem:** Write SMARTS patterns to match:
1. Primary alcohol (R-CH2-OH)
2. Secondary amine (R2-NH)
3. Aromatic nitro group
4. Sulfonamide (-SO2NH2)

Test each on appropriate molecules and print the number of matches found.

**Hints:**
- `[OX2H1][CX4H2]` matches primary alcohols specifically (OH on a CH2).
- `[NX3H1]([#6])[#6]` matches secondary amines.
- `[$(c[N+](=O)[O-])]` or `[$([NX3](=O)=O)]` for aromatic nitro.
- `[SX4](=[OX1])(=[OX1])([NX3H2])` for sulfonamide.
- Use `mol.HasSubstructMatch(pattern)` or `mol.GetSubstructMatches(pattern)`.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 4.1
# Define SMARTS patterns
patterns = {
    "Primary alcohol": "[OX2H1][CX4H2]",
    "Secondary amine": "[NX3H1]([#6])[#6]",
    "Aromatic nitro": "[cH0:1][N+](=O)[O-]",
    "Sulfonamide": "[SX4](=[OX1])(=[OX1])[NX3H2]",
}

# Test molecules for each pattern
test_cases = {
    "Primary alcohol": [("Ethanol", "CCO"), ("Benzyl alcohol", "OCC1=CC=CC=C1"), ("Phenol", "OC1=CC=CC=C1")],
    "Secondary amine": [("Diethylamine", "CCNCC"), ("Trimethylamine", "CN(C)C"), ("Aniline", "NC1=CC=CC=C1")],
    "Aromatic nitro": [("Nitrobenzene", "C1=CC=CC=C1[N+](=O)[O-]"), ("Toluene", "CC1=CC=CC=C1")],
    "Sulfonamide": [("Sulfanilamide", "NC1=CC=C(C=C1)S(N)(=O)=O"), ("Celecoxib", "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F")],
}

for fg_name, smarts in patterns.items():
    pat = Chem.MolFromSmarts(smarts)
    print(f"\n{fg_name} ({smarts}):")
    for mol_name, smi in test_cases[fg_name]:
        mol = Chem.MolFromSmiles(smi)
        matches = mol.GetSubstructMatches(pat)
        status = "MATCH" if matches else "no match"
        print(f"  {mol_name:20s} -> {status} ({len(matches)} hits)")

### Exercise 4.2 — Functional Group Inventory Scanner

**Problem:** Using the SMARTS library below, scan each molecule and create a DataFrame showing which functional groups are present (True/False).

```python
fg_smarts = {
    "Alcohol": "[OX2H1][CX4]",
    "Carboxylic acid": "[CX3](=O)[OX2H1]",
    "Amine (primary)": "[NX3H2]",
    "Amide": "[NX3][CX3](=[OX1])",
    "Ester": "[CX3](=O)[OX2][#6]",
    "Ether": "[OD2]([CX4])[CX4]",
}

test_mols = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Lidocaine": "CCN(CC)CC(=O)NC1=C(C)C=CC=C1C",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Penicillin V": "CC1(C)S[C@@H]2[C@H](NC(=O)COC3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O",
}
```

**Hints:**
- Loop over molecules (outer) and functional groups (inner).
- `mol.HasSubstructMatch(pattern)` returns True/False.
- Build a nested dict or list of dicts, then convert to `pd.DataFrame`.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 4.2
fg_smarts = {
    "Alcohol": "[OX2H1][CX4]",
    "Carboxylic acid": "[CX3](=O)[OX2H1]",
    "Amine (primary)": "[NX3H2]",
    "Amide": "[NX3][CX3](=[OX1])",
    "Ester": "[CX3](=O)[OX2][#6]",
    "Ether": "[OD2]([CX4])[CX4]",
}

test_mols = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Lidocaine": "CCN(CC)CC(=O)NC1=C(C)C=CC=C1C",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Penicillin V": "CC1(C)S[C@@H]2[C@H](NC(=O)COC3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O",
}

# Compile SMARTS patterns
fg_patterns = {name: Chem.MolFromSmarts(sma) for name, sma in fg_smarts.items()}

# Build the inventory
rows = []
for mol_name, smi in test_mols.items():
    mol = Chem.MolFromSmiles(smi)
    row = {"Molecule": mol_name}
    for fg_name, pat in fg_patterns.items():
        row[fg_name] = mol.HasSubstructMatch(pat)
    rows.append(row)

df_fg = pd.DataFrame(rows).set_index("Molecule")
# Display with checkmarks for readability
display(df_fg.replace({True: "Yes", False: "-"}))

### Exercise 4.3 — Find the MCS of Beta-Lactam Antibiotics

**Problem:** Find the maximum common substructure (MCS) shared by these three antibiotics. Visualize the MCS highlighted in each molecule.

```python
antibiotics = {
    "Penicillin V": "CC1(C)S[C@@H]2[C@H](NC(=O)COC3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O",
    "Amoxicillin": "CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)C3=CC=C(O)C=C3)C(=O)N2[C@@H]1C(=O)O",
    "Ampicillin": "CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)C3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O",
}
```

**Hints:**
- `rdFMCS.FindMCS(mol_list)` returns an MCS result with a `.smartsString` attribute.
- Convert the SMARTS to a mol with `Chem.MolFromSmarts()` for highlighting.
- Use `mol.GetSubstructMatch(mcs_mol)` to get atom indices, then pass to `Draw.MolsToGridImage(highlightAtomLists=...)`.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 4.3
antibiotics = {
    "Penicillin V": "CC1(C)S[C@@H]2[C@H](NC(=O)COC3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O",
    "Amoxicillin": "CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)C3=CC=C(O)C=C3)C(=O)N2[C@@H]1C(=O)O",
    "Ampicillin": "CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)C3=CC=CC=C3)C(=O)N2[C@@H]1C(=O)O",
}

mols = [Chem.MolFromSmiles(smi) for smi in antibiotics.values()]
names = list(antibiotics.keys())

# Find MCS
mcs_result = rdFMCS.FindMCS(mols)
print(f"MCS SMARTS: {mcs_result.smartsString}")
print(f"MCS atoms:  {mcs_result.numAtoms}")
print(f"MCS bonds:  {mcs_result.numBonds}")

# Convert MCS to a query molecule for highlighting
mcs_mol = Chem.MolFromSmarts(mcs_result.smartsString)

# Highlight MCS atoms in each molecule
highlight_lists = []
for mol in mols:
    match = mol.GetSubstructMatch(mcs_mol)
    highlight_lists.append(list(match))

Draw.MolsToGridImage(
    mols, molsPerRow=3, legends=names,
    highlightAtomLists=highlight_lists, subImgSize=(350, 300)
)

---

## Topic 5: Fingerprints & Similarity (Notebook 05)

### Exercise 5.1 — Tanimoto Similarity Matrix for NSAIDs

**Problem:** Compute a 6x6 Tanimoto similarity matrix using Morgan fingerprints (radius=2, 2048 bits) and display as a heatmap. Which pair is most similar? Which is least similar?

```python
nsaids = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Naproxen": "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Indomethacin": "COC1=CC2=C(C=C1)C(=C(CC(=O)O)N2C(=O)C1=CC=C(Cl)C=C1)C",
}
```

**Hints:**
- Use `rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)` to create the generator.
- `gen.GetFingerprint(mol)` returns a fingerprint object.
- `DataStructs.TanimotoSimilarity(fp1, fp2)` computes the similarity.
- Build a numpy matrix, then use `sns.heatmap()` with `annot=True`.
- To find the most/least similar pair, mask the diagonal (self-similarity = 1.0).

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 5.1
nsaids = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Naproxen": "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Indomethacin": "COC1=CC2=C(C=C1)C(=C(CC(=O)O)N2C(=O)C1=CC=C(Cl)C=C1)C",
}

names = list(nsaids.keys())
mols = [Chem.MolFromSmiles(smi) for smi in nsaids.values()]

# Generate Morgan fingerprints
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(mol) for mol in mols]

# Build similarity matrix
n = len(fps)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_matrix, xticklabels=names, yticklabels=names,
            annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
ax.set_title("Tanimoto Similarity Matrix (Morgan r=2)")
plt.tight_layout()
plt.show()

# Find most and least similar pairs (excluding diagonal)
mask = np.eye(n, dtype=bool)
sim_masked = np.ma.array(sim_matrix, mask=mask)

max_idx = np.unravel_index(sim_masked.argmax(), sim_matrix.shape)
min_idx = np.unravel_index(sim_masked.argmin(), sim_matrix.shape)
print(f"\nMost similar pair:  {names[max_idx[0]]} & {names[max_idx[1]]} (Tc = {sim_matrix[max_idx]:.3f})")
print(f"Least similar pair: {names[min_idx[0]]} & {names[min_idx[1]]} (Tc = {sim_matrix[min_idx]:.3f})")

### Exercise 5.2 — Compare Fingerprint Types

**Problem:** For aspirin and ibuprofen, compute Tanimoto similarity using four different fingerprint types: Morgan, RDKit, AtomPair, and TopologicalTorsion. Which fingerprint gives the highest similarity? Which gives the lowest?

```python
aspirin = "CC(=O)OC1=CC=CC=C1C(=O)O"
ibuprofen = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
```

**Hints:**
- `rdFingerprintGenerator.GetMorganGenerator()`, `.GetRDKitFPGenerator()`, `.GetAtomPairGenerator()`, `.GetTopologicalTorsionGenerator()`
- Each generator has `.GetFingerprint(mol)`.
- `DataStructs.TanimotoSimilarity()` works on all of them.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 5.2
aspirin = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")
ibuprofen = Chem.MolFromSmiles("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O")

generators = {
    "Morgan (r=2)": rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048),
    "RDKit": rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048),
    "AtomPair": rdFingerprintGenerator.GetAtomPairGenerator(fpSize=2048),
    "TopologicalTorsion": rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=2048),
}

results = {}
for name, gen in generators.items():
    fp1 = gen.GetFingerprint(aspirin)
    fp2 = gen.GetFingerprint(ibuprofen)
    tc = DataStructs.TanimotoSimilarity(fp1, fp2)
    results[name] = tc
    print(f"{name:25s} Tanimoto: {tc:.4f}")

best = max(results, key=results.get)
worst = min(results, key=results.get)
print(f"\nHighest similarity: {best} ({results[best]:.4f})")
print(f"Lowest similarity:  {worst} ({results[worst]:.4f})")

### Exercise 5.3 — Find the Nearest Neighbor

**Problem:** Given naproxen as the query molecule, find its nearest neighbor (most similar) and the most distant molecule in the NSAID set using `DataStructs.BulkTanimotoSimilarity`. Use Morgan fingerprints (radius=2).

```python
query = "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O"  # Naproxen
library = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Indomethacin": "COC1=CC2=C(C=C1)C(=C(CC(=O)O)N2C(=O)C1=CC=C(Cl)C=C1)C",
}
```

**Hints:**
- `DataStructs.BulkTanimotoSimilarity(query_fp, list_of_fps)` returns a list of similarities.
- Use `max()` / `min()` with index tracking, or `np.argmax()` / `np.argmin()`.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 5.3
query_smi = "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O"  # Naproxen
library = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Indomethacin": "COC1=CC2=C(C=C1)C(=C(CC(=O)O)N2C(=O)C1=CC=C(Cl)C=C1)C",
}

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

query_mol = Chem.MolFromSmiles(query_smi)
query_fp = gen.GetFingerprint(query_mol)

lib_names = list(library.keys())
lib_fps = [gen.GetFingerprint(Chem.MolFromSmiles(smi)) for smi in library.values()]

# Bulk similarity
sims = DataStructs.BulkTanimotoSimilarity(query_fp, lib_fps)

print("Naproxen vs. library:")
for name, sim in zip(lib_names, sims):
    print(f"  {name:15s} Tc = {sim:.4f}")

nearest_idx = np.argmax(sims)
farthest_idx = np.argmin(sims)
print(f"\nNearest neighbor:  {lib_names[nearest_idx]} (Tc = {sims[nearest_idx]:.4f})")
print(f"Most distant:      {lib_names[farthest_idx]} (Tc = {sims[farthest_idx]:.4f})")

---

## Topic 6: Chemical Reactions (Notebook 06)

### Exercise 6.1 — Encode Suzuki Coupling and Fischer Esterification

**Problem:**
1. Write a reaction SMARTS for **Suzuki coupling**: Ar-Br + Ar-B(OH)2 --> Ar-Ar
2. Write a reaction SMARTS for **Fischer esterification**: R-COOH + R'-OH --> R-COOR' + H2O

Test each with appropriate reagents and print the product SMILES.

**Hints:**
- Reaction SMARTS syntax: `[reactant1].[reactant2]>>[product]`
- Use `rdChemReactions.ReactionFromSmarts()` to create the reaction.
- `rxn.RunReactants((mol1, mol2))` returns a tuple of product tuples.
- For Suzuki: `[c:1][Br].[c:2][B](O)O>>[c:1][c:2]`
- For esterification: `[C:1](=[O:2])[OH].[OH][C:3]>>[C:1](=[O:2])[O][C:3]`

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 6.1

# --- 1. Suzuki Coupling ---
suzuki = rdChemReactions.ReactionFromSmarts("[c:1][Br].[c:2][B](O)O>>[c:1][c:2]")
# Bromobenzene + phenylboronic acid -> biphenyl
bromo = Chem.MolFromSmiles("c1ccc(Br)cc1")        # bromobenzene
boronic = Chem.MolFromSmiles("c1ccc(B(O)O)cc1")   # phenylboronic acid

products = suzuki.RunReactants((bromo, boronic))
print("Suzuki Coupling: bromobenzene + phenylboronic acid")
for i, prod_set in enumerate(products):
    for p in prod_set:
        Chem.SanitizeMol(p)
        print(f"  Product {i+1}: {Chem.MolToSmiles(p)}")

# --- 2. Fischer Esterification ---
fischer = rdChemReactions.ReactionFromSmarts("[C:1](=[O:2])[OH].[OH][C:3]>>[C:1](=[O:2])[O][C:3]")
# Acetic acid + ethanol -> ethyl acetate
acid = Chem.MolFromSmiles("CC(=O)O")   # acetic acid
alcohol = Chem.MolFromSmiles("CCO")     # ethanol

products = fischer.RunReactants((acid, alcohol))
print("\nFischer Esterification: acetic acid + ethanol")
for i, prod_set in enumerate(products):
    for p in prod_set:
        Chem.SanitizeMol(p)
        print(f"  Product {i+1}: {Chem.MolToSmiles(p)}")

### Exercise 6.2 — Enumerate an Amide Library

**Problem:** Using amide coupling (RCOOH + R'NH2 --> RCONHR'), enumerate all products from these building blocks. Display all products as a grid image.

```python
acids = ["c1ccc(C(=O)O)cc1", "CC(=O)O", "OC(=O)c1ccncc1"]   # benzoic acid, acetic acid, isonicotinic acid
amines = ["NCC", "NC1CCCCC1", "Nc1ccccc1"]                     # ethylamine, cyclohexylamine, aniline
```

**Hints:**
- Amide coupling SMARTS: `[C:1](=[O:2])[OH].[NH2:3]>>[C:1](=[O:2])[NH:3]`
- Use nested loops: for each acid x amine pair, run the reaction.
- Collect unique products using canonical SMILES as keys.
- `Draw.MolsToGridImage()` to visualize.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 6.2
amide_rxn = rdChemReactions.ReactionFromSmarts("[C:1](=[O:2])[OH].[NH2:3]>>[C:1](=[O:2])[NH:3]")

acids = ["c1ccc(C(=O)O)cc1", "CC(=O)O", "OC(=O)c1ccncc1"]
amines = ["NCC", "NC1CCCCC1", "Nc1ccccc1"]

product_mols = []
product_labels = []

for a_smi in acids:
    acid_mol = Chem.MolFromSmiles(a_smi)
    for n_smi in amines:
        amine_mol = Chem.MolFromSmiles(n_smi)
        results = amide_rxn.RunReactants((acid_mol, amine_mol))
        if results:
            prod = results[0][0]
            Chem.SanitizeMol(prod)
            prod_smi = Chem.MolToSmiles(prod)
            product_mols.append(Chem.MolFromSmiles(prod_smi))
            product_labels.append(prod_smi)

print(f"Generated {len(product_mols)} amide products (3 acids x 3 amines)")
Draw.MolsToGridImage(product_mols, molsPerRow=3, legends=product_labels, subImgSize=(300, 250))

---

## Topic 7: 3D Conformers (Notebook 07)

### Exercise 7.1 — Generate Conformer Ensembles and Analyze Energies

**Problem:** For ibuprofen (`"CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"`):
1. Generate 30 conformers with ETKDG and MMFF optimization
2. Report the minimum, maximum, and mean MMFF energy
3. Compute the RMSD between the lowest-energy and highest-energy conformers

**Hints:**
- `Chem.AddHs(mol)` before embedding.
- `AllChem.EmbedMultipleConfs(mol, numConfs=30, params=AllChem.ETKDGv3())` for generation.
- `AllChem.MMFFOptimizeMoleculeConfs(mol)` returns a list of `(converged, energy)` tuples.
- `AllChem.GetConformerRMS(mol, confId1, confId2)` for RMSD.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 7.1
mol = Chem.MolFromSmiles("CC(C)CC1=CC=C(C=C1)C(C)C(=O)O")
mol = Chem.AddHs(mol)

# Generate conformers
params = AllChem.ETKDGv3()
params.randomSeed = 42
cids = AllChem.EmbedMultipleConfs(mol, numConfs=30, params=params)
print(f"Generated {len(cids)} conformers")

# Optimize with MMFF and collect energies
results = AllChem.MMFFOptimizeMoleculeConfs(mol)
energies = [(cid, energy) for cid, (converged, energy) in zip(cids, results) if converged == 0]

energy_values = [e for _, e in energies]
print(f"\nMMFF Energies (kcal/mol):")
print(f"  Min:  {min(energy_values):.2f}")
print(f"  Max:  {max(energy_values):.2f}")
print(f"  Mean: {np.mean(energy_values):.2f}")

# Find lowest and highest energy conformers
min_conf = min(energies, key=lambda x: x[1])
max_conf = max(energies, key=lambda x: x[1])

# Compute RMSD between them
rmsd = AllChem.GetConformerRMS(mol, min_conf[0], max_conf[0])
print(f"\nRMSD between lowest-energy (conf {min_conf[0]}, {min_conf[1]:.2f} kcal/mol)")
print(f"  and highest-energy (conf {max_conf[0]}, {max_conf[1]:.2f} kcal/mol): {rmsd:.3f} A")

### Exercise 7.2 — Align Two NSAIDs and Report RMSD

**Problem:** Generate a 3D conformer for naproxen and ibuprofen, then align naproxen onto ibuprofen using:
1. `AllChem.AlignMol()` (atom-order-based alignment)
2. Open3D alignment via `AllChem.GetO3A()` (shape-based)

Compare the RMSD values from both methods.

```python
naproxen_smi = "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O"
ibuprofen_smi = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
```

**Hints:**
- Embed each molecule with `AllChem.EmbedMolecule()` + `AllChem.MMFFOptimizeMolecule()`.
- For MCS-based alignment: find common substructure with `rdFMCS.FindMCS()`, get atom maps, pass to `AllChem.AlignMol(probe, ref, atomMap=...)`.
- For O3A: `AllChem.GetO3A(probe, ref)` returns an O3A object; call `.Align()` which returns RMSD.
- Make sure to `AddHs` before embedding and `RemoveHs` before MCS if needed.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 7.2
naproxen_smi = "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O"
ibuprofen_smi = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"

def embed_and_optimize(smi):
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)
    return mol

ref = embed_and_optimize(ibuprofen_smi)    # reference
probe = embed_and_optimize(naproxen_smi)   # probe (will be moved)

# Method 1: MCS-based alignment
# Find MCS to get atom mapping
ref_noH = Chem.RemoveHs(ref)
probe_noH = Chem.RemoveHs(probe)
mcs = rdFMCS.FindMCS([ref_noH, probe_noH])
mcs_mol = Chem.MolFromSmarts(mcs.smartsString)

ref_match = ref_noH.GetSubstructMatch(mcs_mol)
probe_match = probe_noH.GetSubstructMatch(mcs_mol)
atom_map = list(zip(probe_match, ref_match))

# We need to work with the H-containing mols for 3D coords
# but use the heavy-atom mapping
rmsd_align = AllChem.AlignMol(probe, ref, atomMap=atom_map)
print(f"Method 1 (MCS-based AlignMol):  RMSD = {rmsd_align:.3f} A")

# Method 2: O3A alignment (re-embed probe to start fresh)
probe2 = embed_and_optimize(naproxen_smi)
pyMMFF_ref = AllChem.MMFFGetMoleculeProperties(ref)
pyMMFF_probe = AllChem.MMFFGetMoleculeProperties(probe2)
o3a = AllChem.GetO3A(probe2, ref, pyMMFF_probe, pyMMFF_ref)
rmsd_o3a = o3a.Align()
print(f"Method 2 (O3A shape alignment):  RMSD = {rmsd_o3a:.3f} A")

---

## Topic 8: Molecular Datasets (Notebook 08)

### Exercise 8.1 — Load, Curate, and Compute Properties

**Problem:** Given this SMILES list (with some intentionally dirty data), build a curation pipeline:

```python
raw_smiles = [
    ("Drug A", "CCO"),
    ("Drug B", "CC(=O)[O-].[Na+]"),      # salt
    ("Drug C", "invalid_smiles"),           # invalid
    ("Drug D", "c1ccccc1"),                 # benzene
    ("Drug E", "c1ccccc1"),                 # duplicate
    ("Drug F", "[NH3+]CCC(=O)[O-]"),       # zwitterion (beta-alanine)
    ("Drug G", "CC(=O)OC1=CC=CC=C1C(=O)O"), # aspirin
]
```

Steps:
1. Parse SMILES and remove invalid entries
2. Strip salts (keep largest fragment)
3. Neutralize charges
4. Deduplicate by canonical SMILES
5. Compute MW and LogP for the curated set

**Hints:**
- Salt stripping: split on `.` and keep the fragment with the most heavy atoms, or use `rdMolStandardize.LargestFragmentChooser()`.
- Neutralization: use `rdMolStandardize.Uncharger()` from `rdkit.Chem.MolStandardize`.
- Deduplication: convert to canonical SMILES and use a set/dict to track seen molecules.
- Track how many molecules survive each step.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 8.1
from rdkit.Chem.MolStandardize import rdMolStandardize

raw_smiles = [
    ("Drug A", "CCO"),
    ("Drug B", "CC(=O)[O-].[Na+]"),
    ("Drug C", "invalid_smiles"),
    ("Drug D", "c1ccccc1"),
    ("Drug E", "c1ccccc1"),
    ("Drug F", "[NH3+]CCC(=O)[O-]"),
    ("Drug G", "CC(=O)OC1=CC=CC=C1C(=O)O"),
]

# Step 1: Parse and remove invalid
print(f"Starting with {len(raw_smiles)} entries")
parsed = []
for name, smi in raw_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        parsed.append((name, mol))
    else:
        print(f"  Removed (invalid): {name} -> {smi}")
print(f"After parsing: {len(parsed)}")

# Step 2: Strip salts (keep largest fragment)
chooser = rdMolStandardize.LargestFragmentChooser()
stripped = []
for name, mol in parsed:
    mol = chooser.choose(mol)
    stripped.append((name, mol))
print(f"After salt stripping: {len(stripped)}")

# Step 3: Neutralize charges
uncharger = rdMolStandardize.Uncharger()
neutralized = []
for name, mol in stripped:
    mol = uncharger.uncharge(mol)
    neutralized.append((name, mol))
print(f"After neutralization: {len(neutralized)}")

# Step 4: Deduplicate by canonical SMILES
seen = {}
deduplicated = []
for name, mol in neutralized:
    can_smi = Chem.MolToSmiles(mol)
    if can_smi not in seen:
        seen[can_smi] = name
        deduplicated.append((name, mol, can_smi))
    else:
        print(f"  Removed (duplicate of {seen[can_smi]}): {name}")
print(f"After deduplication: {len(deduplicated)}")

# Step 5: Compute properties
rows = []
for name, mol, can_smi in deduplicated:
    rows.append({
        "Name": name,
        "SMILES": can_smi,
        "MW": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
    })

df_curated = pd.DataFrame(rows).set_index("Name")
print(f"\nCurated dataset ({len(df_curated)} molecules):")
display(df_curated)

### Exercise 8.2 — Write Curated Data to SDF and Read It Back

**Problem:** Take these curated molecules, add computed MW and LogP as SD tags, write to a temporary SDF file, then read it back and verify the properties round-trip correctly.

```python
curated = {
    "Ethanol": "CCO",
    "Acetic acid": "CC(=O)O",
    "Benzene": "c1ccccc1",
    "Beta-alanine": "NCCC(=O)O",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
}
```

**Hints:**
- Use `Chem.SDWriter(filename)` to write. Set properties with `mol.SetProp("MW", str(value))`.
- You need 2D/3D coords: use `AllChem.Compute2DCoords(mol)` before writing.
- Read back with `Chem.SDMolSupplier(filename)`.
- Verify with `mol.GetProp("MW")` on the read-back molecules.
- Use `tempfile.NamedTemporaryFile(suffix=".sdf")` or a fixed temp path.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 8.2
import tempfile, os

curated = {
    "Ethanol": "CCO",
    "Acetic acid": "CC(=O)O",
    "Benzene": "c1ccccc1",
    "Beta-alanine": "NCCC(=O)O",
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
}

# Write to SDF
sdf_path = os.path.join(tempfile.gettempdir(), "curated_exercise.sdf")
writer = Chem.SDWriter(sdf_path)

for name, smi in curated.items():
    mol = Chem.MolFromSmiles(smi)
    AllChem.Compute2DCoords(mol)
    mol.SetProp("_Name", name)
    mol.SetProp("MW", f"{Descriptors.MolWt(mol):.2f}")
    mol.SetProp("LogP", f"{Descriptors.MolLogP(mol):.2f}")
    writer.write(mol)
writer.close()
print(f"Wrote {len(curated)} molecules to {sdf_path}")

# Read back and verify
supplier = Chem.SDMolSupplier(sdf_path)
print(f"\nRead back {len(supplier)} molecules:\n")
print(f"{'Name':15s} | {'MW (stored)':>11s} | {'MW (recomputed)':>15s} | {'Match':>5s}")
print("-" * 60)
for mol in supplier:
    if mol is None:
        continue
    name = mol.GetProp("_Name")
    mw_stored = mol.GetProp("MW")
    mw_recomputed = f"{Descriptors.MolWt(mol):.2f}"
    match = "OK" if mw_stored == mw_recomputed else "FAIL"
    print(f"{name:15s} | {mw_stored:>11s} | {mw_recomputed:>15s} | {match:>5s}")

# Clean up
os.remove(sdf_path)

---

## Topic 9: Drug-Likeness Filters (Notebook 09)

### Exercise 9.1 — Multi-Filter Pipeline with Failure Analysis

**Problem:** Apply Lipinski, Veber, and PAINS filters to these molecules. For each molecule that fails any filter, explain **why** it fails. Create a summary DataFrame.

```python
test_compounds = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Cyclosporine": "CCC1NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C)NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C(C)C)N(C)C(=O)C(CC(C)C)N(C)C1=O",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Atorvastatin": "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
}
```

**Filter criteria:**
- **Lipinski Ro5:** MW <= 500, LogP <= 5, HBD <= 5, HBA <= 10
- **Veber:** TPSA <= 140, RotBonds <= 10
- **PAINS:** Use `FilterCatalog` with PAINS filter

**Hints:**
- For Lipinski/Veber: compute descriptors and check thresholds manually.
- For PAINS: `FilterCatalog.FilterCatalogParams(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)` then `FilterCatalog.FilterCatalog(params)`.
- `catalog.HasMatch(mol)` returns True if the molecule matches any PAINS pattern.
- Build a list of failure reasons per molecule.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 9.1
test_compounds = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Cyclosporine": "CCC1NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C)NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C(C)C)N(C)C(=O)C(CC(C)C)N(C)C1=O",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Atorvastatin": "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
}

# Set up PAINS filter
pains_params = FilterCatalog.FilterCatalogParams()
pains_params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(pains_params)

rows = []
for name, smi in test_compounds.items():
    mol = Chem.MolFromSmiles(smi)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rotb = rdMolDescriptors.CalcNumRotatableBonds(mol)

    # Lipinski
    lip_violations = []
    if mw > 500: lip_violations.append(f"MW={mw:.0f}>500")
    if logp > 5: lip_violations.append(f"LogP={logp:.1f}>5")
    if hbd > 5: lip_violations.append(f"HBD={hbd}>5")
    if hba > 10: lip_violations.append(f"HBA={hba}>10")
    lipinski_pass = len(lip_violations) <= 1  # Ro5 allows 1 violation

    # Veber
    veber_violations = []
    if tpsa > 140: veber_violations.append(f"TPSA={tpsa:.0f}>140")
    if rotb > 10: veber_violations.append(f"RotB={rotb}>10")
    veber_pass = len(veber_violations) == 0

    # PAINS
    pains_pass = not pains_catalog.HasMatch(mol)

    failures = []
    if not lipinski_pass: failures.append(f"Lipinski ({', '.join(lip_violations)})")
    if not veber_pass: failures.append(f"Veber ({', '.join(veber_violations)})")
    if not pains_pass: failures.append("PAINS match")

    rows.append({
        "Name": name,
        "MW": round(mw, 1),
        "LogP": round(logp, 2),
        "HBD": hbd, "HBA": hba,
        "TPSA": round(tpsa, 1),
        "RotB": rotb,
        "Lipinski": "Pass" if lipinski_pass else "FAIL",
        "Veber": "Pass" if veber_pass else "FAIL",
        "PAINS": "Pass" if pains_pass else "FAIL",
        "Failure reasons": "; ".join(failures) if failures else "None",
    })

df_filters = pd.DataFrame(rows).set_index("Name")
display(df_filters)

### Exercise 9.2 — Compare How Different Filters Classify the Same Molecules

**Problem:** For the same molecules, apply Lipinski, Ghose, Egan, and Muegge filters. Create a pass/fail DataFrame. Do the filters always agree?

**Filter criteria:**
- **Lipinski:** MW <= 500, LogP <= 5, HBD <= 5, HBA <= 10
- **Ghose:** 160 <= MW <= 480, -0.4 <= LogP <= 5.6, 40 <= atoms <= 130, 20 <= MR <= 70
- **Egan:** LogP <= 5.88, TPSA <= 131.6
- **Muegge:** 200 <= MW <= 600, -2 <= LogP <= 5, TPSA <= 150, RotBonds <= 15, HBD <= 5, HBA <= 10, rings <= 7

```python
test_compounds = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Cyclosporine": "CCC1NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C)NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C(C)C)N(C)C(=O)C(CC(C)C)N(C)C1=O",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Atorvastatin": "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
}
```

**Hints:**
- Compute all descriptors once per molecule, then evaluate each filter.
- `Descriptors.MolMR(mol)` for molar refractivity (Ghose).
- `mol.GetNumHeavyAtoms()` for atom count (Ghose uses total atoms including H).
- `rdMolDescriptors.CalcNumRings(mol)` for ring count (Muegge).
- Present results as a colored DataFrame for easy comparison.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 9.2
test_compounds = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Cyclosporine": "CCC1NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C)NC(=O)C(CC(C)C)N(C)C(=O)C(CC(C)C)N(C)C(=O)C(C)NC(=O)C(C(C)C)N(C)C(=O)C(CC(C)C)N(C)C1=O",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Atorvastatin": "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1",
}

def apply_filters(mol):
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rotb = rdMolDescriptors.CalcNumRotatableBonds(mol)
    mr = Descriptors.MolMR(mol)
    natoms = mol.GetNumAtoms()  # heavy atoms in RDKit; Ghose uses total
    nrings = rdMolDescriptors.CalcNumRings(mol)

    # Lipinski (allow 1 violation)
    lip_v = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    lipinski = lip_v <= 1

    # Ghose
    ghose = all([160 <= mw <= 480, -0.4 <= logp <= 5.6, 40 <= natoms <= 130, 20 <= mr <= 70])

    # Egan
    egan = all([logp <= 5.88, tpsa <= 131.6])

    # Muegge
    muegge = all([200 <= mw <= 600, -2 <= logp <= 5, tpsa <= 150,
                  rotb <= 15, hbd <= 5, hba <= 10, nrings <= 7])

    return {"Lipinski": lipinski, "Ghose": ghose, "Egan": egan, "Muegge": muegge}

rows = []
for name, smi in test_compounds.items():
    mol = Chem.MolFromSmiles(smi)
    result = apply_filters(mol)
    result["Name"] = name
    rows.append(result)

df_compare = pd.DataFrame(rows).set_index("Name")
df_display = df_compare.replace({True: "Pass", False: "FAIL"})
display(df_display)

# Check agreement
print("\nDo filters always agree?")
for name in df_compare.index:
    vals = df_compare.loc[name].values
    if len(set(vals)) > 1:
        print(f"  {name}: DISAGREE - {dict(df_display.loc[name])}")
    else:
        print(f"  {name}: All agree ({df_display.loc[name].iloc[0]})")

---

## Topic 10: Full Workflow (Notebook 10)

### Exercise 10.1 — End-to-End Pipeline: Similarity Search, Filter, Cluster, Diversity Pick

**Problem:** Build a complete cheminformatics pipeline:
1. Define ibuprofen as the query
2. Search the library for molecules with Tanimoto >= 0.2 (Morgan, radius=2)
3. Apply Lipinski filter to the hits
4. Cluster passing molecules with Butina (distance cutoff=0.6)
5. Pick 5 diverse compounds using MaxMinPicker

Report how many molecules survive each stage. Show the final 5 picks.

```python
library = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Naproxen": "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Omeprazole": "COC1=CC2=C(C=C1C)NC(=N2)S(=O)CC1=CC=C(OC)C(C)=N1",
    "Losartan": "CCCCC1=NC(Cl)=C(N1CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)CO",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Propranolol": "CC(C)NCC(O)COC1=CC=CC2=CC=CC=C12",
    "Fluoxetine": "CNCCC(OC1=CC=C(C(F)(F)F)C=C1)C1=CC=CC=C1",
}
query_smi = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"  # Ibuprofen
```

**Hints:**
- Similarity search: compute Morgan FPs, use `DataStructs.BulkTanimotoSimilarity()`.
- Lipinski filter: check MW, LogP, HBD, HBA thresholds.
- Butina clustering: compute a distance matrix (1 - Tanimoto), then `Butina.ClusterData()`.
- MaxMinPicker: `rdSimDivPickers.MaxMinPicker()`, call `.LazyBitVectorPick(fps, n_pool, n_picks)`.
- Track counts at each stage for the pipeline report.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 10.1
library = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Naproxen": "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Omeprazole": "COC1=CC2=C(C=C1C)NC(=N2)S(=O)CC1=CC=C(OC)C(C)=N1",
    "Losartan": "CCCCC1=NC(Cl)=C(N1CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)CO",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Propranolol": "CC(C)NCC(O)COC1=CC=CC2=CC=CC=C12",
    "Fluoxetine": "CNCCC(OC1=CC=C(C(F)(F)F)C=C1)C1=CC=CC=C1",
}
query_smi = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"  # Ibuprofen

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
query_mol = Chem.MolFromSmiles(query_smi)
query_fp = gen.GetFingerprint(query_mol)

# Build library mols and fps
lib_names = list(library.keys())
lib_mols = [Chem.MolFromSmiles(smi) for smi in library.values()]
lib_fps = [gen.GetFingerprint(mol) for mol in lib_mols]

print(f"Library size: {len(lib_mols)}")

# --- Step 1: Similarity search (Tc >= 0.2) ---
sims = DataStructs.BulkTanimotoSimilarity(query_fp, lib_fps)
hits = [(name, mol, fp, tc) for name, mol, fp, tc in zip(lib_names, lib_mols, lib_fps, sims) if tc >= 0.2]
print(f"After similarity search (Tc >= 0.2): {len(hits)} hits")

# --- Step 2: Lipinski filter ---
def passes_lipinski(mol):
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    violations = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    return violations <= 1

filtered = [(name, mol, fp, tc) for name, mol, fp, tc in hits if passes_lipinski(mol)]
print(f"After Lipinski filter: {len(filtered)} compounds")

# --- Step 3: Butina clustering ---
filtered_fps = [fp for _, _, fp, _ in filtered]
filtered_names = [name for name, _, _, _ in filtered]
n = len(filtered_fps)

# Build distance matrix (lower triangular)
dists = []
for i in range(1, n):
    for j in range(i):
        dist = 1.0 - DataStructs.TanimotoSimilarity(filtered_fps[i], filtered_fps[j])
        dists.append(dist)

clusters = Butina.ClusterData(dists, n, 0.6, isDistData=True)
print(f"Butina clustering: {len(clusters)} clusters")
for i, cluster in enumerate(clusters):
    members = [filtered_names[idx] for idx in cluster]
    print(f"  Cluster {i+1}: {members}")

# --- Step 4: MaxMinPicker for diversity ---
n_picks = min(5, len(filtered_fps))
picker = rdSimDivPickers.MaxMinPicker()
pick_indices = picker.LazyBitVectorPick(filtered_fps, len(filtered_fps), n_picks)
picked = [(filtered[i][0], filtered[i][1]) for i in pick_indices]

print(f"\nFinal {n_picks} diverse picks:")
for name, mol in picked:
    print(f"  {name}")

# Display the picked molecules
Draw.MolsToGridImage(
    [mol for _, mol in picked],
    legends=[name for name, _ in picked],
    molsPerRow=min(5, n_picks), subImgSize=(300, 250)
)

### Exercise 10.2 — Scaffold Analysis and Chemical Space Visualization

**Problem:** For the library above:
1. Extract Murcko scaffolds for each molecule
2. How many unique scaffolds are there?
3. Visualize all molecules in 2D using PCA on Morgan fingerprints, colored by scaffold cluster

```python
library = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Naproxen": "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Omeprazole": "COC1=CC2=C(C=C1C)NC(=N2)S(=O)CC1=CC=C(OC)C(C)=N1",
    "Losartan": "CCCCC1=NC(Cl)=C(N1CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)CO",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Propranolol": "CC(C)NCC(O)COC1=CC=CC2=CC=CC=C12",
    "Fluoxetine": "CNCCC(OC1=CC=C(C(F)(F)F)C=C1)C1=CC=CC=C1",
}
```

**Hints:**
- `MurckoScaffold.GetScaffoldForMol(mol)` returns the Murcko scaffold.
- `Chem.MolToSmiles(scaffold)` to get canonical scaffold SMILES for grouping.
- For PCA: convert fingerprints to numpy arrays, fit PCA(n_components=2), then scatter plot.
- Color by scaffold: assign a numeric scaffold ID and use a colormap.

In [ ]:
# Your solution here

<details>
<summary><b>Reference Solution (click to reveal)</b></summary>

Run the code cell below to see the reference solution.
</details>

In [ ]:
# Reference Solution 10.2
library = {
    "Aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "Naproxen": "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O",
    "Diclofenac": "OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl",
    "Celecoxib": "CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F",
    "Acetaminophen": "CC(=O)NC1=CC=C(O)C=C1",
    "Caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Metformin": "CN(C)C(=N)NC(=N)N",
    "Omeprazole": "COC1=CC2=C(C=C1C)NC(=N2)S(=O)CC1=CC=C(OC)C(C)=N1",
    "Losartan": "CCCCC1=NC(Cl)=C(N1CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)CO",
    "Gabapentin": "NCC1(CC(=O)O)CCCCC1",
    "Propranolol": "CC(C)NCC(O)COC1=CC=CC2=CC=CC=C12",
    "Fluoxetine": "CNCCC(OC1=CC=C(C(F)(F)F)C=C1)C1=CC=CC=C1",
}

names = list(library.keys())
mols = [Chem.MolFromSmiles(smi) for smi in library.values()]

# --- Step 1 & 2: Extract Murcko scaffolds ---
scaffolds = []
scaffold_smiles = []
for mol in mols:
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    scaf_smi = Chem.MolToSmiles(scaf)
    scaffolds.append(scaf)
    scaffold_smiles.append(scaf_smi)

unique_scaffolds = list(set(scaffold_smiles))
print(f"Total molecules: {len(mols)}")
print(f"Unique Murcko scaffolds: {len(unique_scaffolds)}")

# Print scaffold assignments
scaffold_to_id = {s: i for i, s in enumerate(unique_scaffolds)}
scaffold_ids = [scaffold_to_id[s] for s in scaffold_smiles]

for name, scaf_smi, sid in zip(names, scaffold_smiles, scaffold_ids):
    print(f"  {name:15s} -> Scaffold {sid}: {scaf_smi}")

# --- Step 3: PCA visualization ---
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(mol) for mol in mols]

# Convert to numpy array
fp_array = np.zeros((len(fps), 2048))
for i, fp in enumerate(fps):
    arr = np.zeros(2048)
    DataStructs.ConvertToNumpyArray(fp, arr)
    fp_array[i] = arr

# PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(fp_array)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
cmap = plt.cm.get_cmap("tab10", len(unique_scaffolds))

for i, (name, x, y, sid) in enumerate(zip(names, coords[:, 0], coords[:, 1], scaffold_ids)):
    ax.scatter(x, y, c=[cmap(sid)], s=100, edgecolors="k", zorder=2)
    ax.annotate(name, (x, y), fontsize=8, ha="left", va="bottom",
                xytext=(5, 5), textcoords="offset points")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Chemical Space: PCA on Morgan FPs, colored by Murcko scaffold")

# Legend for scaffolds
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=cmap(i), edgecolor="k", label=f"Scaffold {i}: {s[:25]}...")
                   for i, s in enumerate(unique_scaffolds)]
ax.legend(handles=legend_elements, loc="best", fontsize=7, title="Scaffolds")
plt.tight_layout()
plt.show()